# Multi-Asset Portfolio & Active Fund Research

**Question:** Can a constrained, minimum-variance allocation reduce client portfolio risk relative to a policy with the same equity budget? How do two active fixed-income funds compare with investable benchmark proxies?

Real historical adjusted market prices, January 2014–December 2025. Initial 36 months train the allocation model; holding-period evaluation begins January 2017. This is retrospective fund-level research, not individual-manager attribution or an investment recommendation.

First run `python -m multi_asset.pipeline --download`. This notebook reads validated local outputs and never substitutes simulated data.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from multi_asset.config import ROOT, INVESTABLE, PROFILES
from multi_asset.data import load, monthly_prices
from multi_asset.analytics import drawdown
from multi_asset.portfolio import optimize
plt.rcParams.update({'figure.figsize': (11, 4), 'axes.spines.top': False, 'axes.spines.right': False})
prices, macro, manifest = load()
monthly = monthly_prices(prices)
returns = monthly.pct_change(fill_method=None).iloc[1:]
folder = ROOT / 'data/processed'
def read(name):
    return pd.read_csv(folder / f'{name}.csv')
facts = json.loads((ROOT / 'reports/local/research_facts.json').read_text())
display(pd.Series(facts, name='Research facts').to_frame())

## 1. Provenance and data quality

Yahoo adjusted close accounts for splits and distributions and is used as a total-return proxy. Market prices differ from NAV, and adjusted history may be revised. Data are USD-denominated, including unhedged foreign equity exposure. Provider payloads and normalized files have SHA-256 checksums. Missing or stale month-end prices cause failure; there is no forward filling.

FRED DGS10/DGS2 are Treasury yields in percent, not bond returns. Current downloaded history is used only for descriptive analysis, not vintage-sensitive trading signals.

In [ ]:
quality = pd.DataFrame({
    'daily_observations': prices.groupby('ticker').size(),
    'first_observation': prices.groupby('ticker').date.min(),
    'last_observation': prices.groupby('ticker').date.max(),
    'monthly_observations': monthly.count(),
    'missing_monthly': monthly.isna().sum()
})
display(quality)
assert not prices.duplicated(['date', 'ticker']).any()
assert not monthly.isna().any().any()
assert (prices.adjusted_close > 0).all()
display(pd.DataFrame(manifest).dropna(subset=['retrieved_utc'])[['series', 'source', 'retrieved_utc', 'rows']])

## 2. Exploratory return and diversification analysis

Normalizing wealth to 100 permits comparison, but does not remove differing risks or imply equal suitability. Correlations are full-sample descriptions; only trailing observations enter portfolio optimization.

In [ ]:
wealth = 100 * (1 + returns[INVESTABLE]).cumprod()
wealth = pd.concat([pd.DataFrame(100., index=[monthly.index[0]], columns=INVESTABLE), wealth])
ax = wealth.plot(title='Historical growth of 100 · adjusted market-price proxy')
ax.set_ylabel('USD index level'); plt.tight_layout(); plt.show()
summary = pd.DataFrame({'annual_mean_return': returns[INVESTABLE].mean()*12,
                        'annual_volatility': returns[INVESTABLE].std()*np.sqrt(12),
                        'worst_month': returns[INVESTABLE].min(),
                        'best_month': returns[INVESTABLE].max()})
display(summary.round(4))

In [ ]:
corr = returns[INVESTABLE].corr()
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(corr)), corr.columns, rotation=45)
ax.set_yticks(range(len(corr)), corr.columns)
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f'{corr.iloc[i,j]:.2f}', ha='center', va='center', fontsize=9)
fig.colorbar(im, ax=ax); ax.set_title('Full-sample monthly return correlations')
plt.tight_layout(); plt.show()

## 3. Client mandates and portfolio construction

Conservative, Balanced and Growth have equity budgets of 25%, 55% and 80%. CVXPY minimizes variance using a 36-month Ledoit–Wolf covariance estimate. All positions are long only and fully invested; each asset is capped at 50%, gold at 10%, and bills at 15%.

For each holding month, training ends in the preceding month. Both policy and optimized portfolios rebalance monthly. The policy splits equities 60/25/15 across SPY/EFA/EEM, holds 5% each in gold and bills, and splits the remaining bond allocation 70/30 across IEF/LQD. Costs are 10bp per dollar bought **or** sold, including the initial purchase. Positions drift before the next rebalance. Month-end execution is idealized.

In [ ]:
portfolio = read('portfolio_returns')
portfolio['date'] = pd.to_datetime(portfolio.date)
portfolio['training_end'] = pd.to_datetime(portfolio.training_end)
assert (portfolio.training_end < portfolio.date).all()
stats = read('portfolio_metrics').set_index('portfolio')
display(stats[['cagr','annual_volatility','maximum_drawdown','tracking_error','annual_traded_fraction']].round(4))
net = portfolio.pivot(index='date', columns='portfolio', values='net_return')
curves = 100*(1+net).cumprod()
curves = pd.concat([pd.DataFrame(100., index=[net.index[0]-pd.offsets.MonthEnd(1)],columns=net.columns), curves])
curves.plot(title='Out-of-sample growth after transaction costs')
plt.ylabel('Growth of 100'); plt.tight_layout(); plt.show()

In [ ]:
net.apply(drawdown).plot(title='Monthly portfolio drawdown')
plt.ylabel('Drawdown (decimal)'); plt.tight_layout(); plt.show()
weights = read('weights')
assert np.allclose(weights.groupby(['date','portfolio']).weight.sum(),1,atol=1e-6)
latest = weights[weights.date == weights.date.max()].pivot(index='ticker',columns='portfolio',values='weight')
display(latest.round(4))
latest.filter(like='Optimized').T.plot.bar(stacked=True,title='Last historical rebalance allocations')
plt.ylabel('Weight'); plt.xticks(rotation=0); plt.tight_layout(); plt.show()

## 4. Historical stress windows

These are realized historical windows, not scenario forecasts. Dates are selected explicitly for descriptive analysis; the optimization did not use event labels. The 2020 window covers February–April and the 2022 window covers the full year. A monthly series cannot reveal the worst intramonth loss.

In [ ]:
events = {'2020 disruption': ('2020-02-01','2020-04-30'), '2022 rate shock': ('2022-01-01','2022-12-31')}
event_returns = pd.DataFrame({name:(1+net.loc[start:end]).prod()-1 for name,(start,end) in events.items()})
display(event_returns.round(4))
event_returns.plot.bar(title='Historical event-window compounded returns')
plt.ylabel('Return (decimal)'); plt.xticks(rotation=25,ha='right'); plt.tight_layout(); plt.show()

## 5. Fund evaluation and factor research

BOND is compared with AGG; MINT with BIL. These ETFs are **investable proxies**, not the funds' official benchmark indexes. MINT can take credit/liquidity/duration risks beyond Treasury bills. Expenses are reflected in fund prices; investor taxes and transaction costs are not applied to this fund-only comparison.

OLS uses excess returns over BIL with HAC standard errors (three lags). When the comparator is BIL itself, benchmark excess returns are identically zero; MINT's benchmark alpha and beta are intentionally undefined. Annual mean active return is still measurable. Factor regressions separately examine SPY−BIL, IEF−BIL and LQD−IEF. The credit-relative factor is not duration-neutral, and none of these coefficients establishes causation or personal manager skill.

In [ ]:
funds = read('fund_metrics').set_index('fund')
display(funds.round(4))
rolling = read('rolling_fund_metrics')
rolling.pivot(index='date', columns='fund', values='information_ratio').plot(title='Rolling 36-month information ratio')
plt.axhline(0,color='black',linewidth=.7); plt.tight_layout(); plt.show()
factors = read('factor_coefficients')
display(factors.round(4))

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(12,4))
for ax,fund in zip(axes,['BOND','MINT']):
    f = factors[(factors.fund==fund)&(factors.term!='const')]
    errors = np.vstack([f.coefficient-f.ci_low, f.ci_high-f.coefficient])
    ax.bar(f.term,f.coefficient,yerr=errors,capsize=4,color='#0e766e')
    ax.axhline(0,color='black',linewidth=.7); ax.set_title(f'{fund}: factor exposures, HAC 95% CI')
    ax.tick_params(axis='x',rotation=20)
plt.tight_layout(); plt.show()
display(read('rate_sensitivity').round(4))
read('monthly_macro').set_index('date')[['DGS10','DGS2']].plot(title='Treasury yields · percentage points')
plt.ylabel('Percent'); plt.tight_layout(); plt.show()

## 6. Walk-forward volatility forecasting

Target: next calendar month's SPY daily log-return standard deviation, annualized with √252. Predictors are lagged realized volatility, three- and twelve-month lagged means, and absolute previous-month return. OLS and a fixed-parameter Random Forest are compared with persistence and a trailing-mean baseline.

After 36 training feature rows, the model is refitted using only matured historical labels. No random split or test-set tuning is used. MAE/RMSE are in annualized-volatility decimal units. Forecasts do not enter the allocation backtest; lowest observed error is not proof of durable superiority.

In [ ]:
predictions = read('volatility_predictions')
assert (pd.to_datetime(predictions.training_end)<pd.to_datetime(predictions.date)).all()
scores = read('volatility_scores').set_index('model')
display(scores.round(4))
scores[['mae','rmse']].plot.bar(title='Walk-forward forecast error · lower is better')
plt.ylabel('Annualized volatility decimal'); plt.xticks(rotation=0); plt.tight_layout(); plt.show()
observed = predictions.drop_duplicates('date').set_index('date').actual
chart = predictions.pivot(index='date',columns='model',values='prediction')
chart['Actual'] = observed
chart.plot(title='SPY realized volatility and one-month-ahead predictions',alpha=.8)
plt.tight_layout(); plt.show()

## 7. Bespoke allocation sensitivity

This is a latest-window optimization exercise, not an additional backtest. Change the equity budget and asset cap to inspect constraint sensitivity. Estimated volatility is in-sample and may understate future risk.

In [ ]:
sensitivity=[]
for equity in [.25,.55,.80]:
    for cap in [.35,.50]:
        w,cov=optimize(returns.tail(36),equity,cap)
        sensitivity.append({'equity_budget':equity,'asset_cap':cap,'estimated_volatility':np.sqrt(w.to_numpy()@cov@w.to_numpy()*12),**w.to_dict()})
display(pd.DataFrame(sensitivity).round(4))

## 8. Findings and limits

Read the generated brief below. Compare optimized portfolios with their matching policies rather than choosing the highest CAGR across different equity budgets. Alpha confidence, benchmark mismatch, factor exposures and trading assumptions must be considered together.

The universe was selected retrospectively and contains survivors. The data provider may revise adjusted histories; saved checksums make the local run auditable but do not create point-in-time data. We do not model leverage, tax, market impact, FX hedges or individual-manager tenure. Current macro observations support retrospective attribution only. Public summaries do not redistribute the raw market-data cache.

In [ ]:
from IPython.display import Markdown
display(Markdown((ROOT/'reports/local/executive_summary.md').read_text()))